# Netflix Movies and TV Shows — Data Cleaning

## Objective

Prepare the Netflix titles dataset for reliable analysis by assessing data quality, investigating missing values, standardizing inconsistent data types, and documenting column-level cleaning decisions.

## Tools

* Python
* Pandas
* Jupyter Notebook

## Deliverable

A cleaned CSV dataset, supported by a documented and reproducible data-cleaning workflow.


In [22]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

In [23]:
data_path = Path("raw_data/netflix_titles.csv")

df = pd.read_csv(data_path)

df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno ...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swi..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noure...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adventure","To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robb..."
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down among the incarcerated women at the Orleans Justice C..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam Khan, Ahsaas Channa, Revathi Pillai, Urvi Singh, Ar...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV Comedies","In a city of coaching centers known to train India’s finest collegiate minds, an earnest but une..."


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [25]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
show_id,8807,8807,s1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,8807,2,Movie,6131,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,8807,8807,Dick Johnson Is Dead,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
director,6173,4528,Rajiv Chilaka,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cast,7982,7692,David Attenborough,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,7976,748,United States,2818,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_added,8797,1767,"January 1, 2020",109,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_year,8807.0,NaN,NaN,NaN,2014.180198,8.819312,1925.0,2013.0,2017.0,2019.0,2021.0
rating,8803,17,TV-MA,3207,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration,8804,220,1 Season,1793,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Duplicate rows: {df.duplicated().sum():,}")

quality_report = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
}).sort_values("missing_percent", ascending=False)

quality_report

Rows: 8,807
Columns: 12
Duplicate rows: 0


,data_type,missing_count,missing_percent,unique_values
director,str,2634,29.91,4528
country,str,831,9.44,748
cast,str,825,9.37,7692
date_added,str,10,0.11,1767
rating,str,4,0.05,17
duration,str,3,0.03,220
show_id,str,0,0.00,8807
type,str,0,0.00,2
title,str,0,0.00,8807
release_year,int64,0,0.00,74


This gives us an evidence-based view of the missing values before making any changes

1. Missing-Value Assessment

Missing values are assessed by column before any cleaning is applied. Descriptive fields such as director, cast, and country may be unavailable in the source data, so their missing values should not automatically cause records to be deleted. Date, rating, and duration fields require additional investigation because they affect temporal and content analysis.

In [27]:
missing_columns = df.columns[df.isna().any()].tolist()

for column in missing_columns:
    print(f"\n{column}: {df[column].isna().sum()} missing values")
    display(df.loc[df[column].isna(), ["show_id", "type", "title", column]].head(5))


director: 2634 missing values


,show_id,type,title,director
1,s2,TV Show,Blood & Water,NaN
3,s4,TV Show,Jailbirds New Orleans,NaN
4,s5,TV Show,Kota Factory,NaN
10,s11,TV Show,"Vendetta: Truth, Lies and The Mafia",NaN
14,s15,TV Show,Crime Stories: India Detectives,NaN



cast: 825 missing values


,show_id,type,title,cast
0,s1,Movie,Dick Johnson Is Dead,NaN
3,s4,TV Show,Jailbirds New Orleans,NaN
10,s11,TV Show,"Vendetta: Truth, Lies and The Mafia",NaN
14,s15,TV Show,Crime Stories: India Detectives,NaN
16,s17,Movie,Europe's Most Dangerous Man: Otto Skorzeny in Spain,NaN



country: 831 missing values


...

,show_id,type,title,date_added
6066,s6067,TV Show,A Young Doctor's Notebook and Other Stories,NaN
6174,s6175,TV Show,Anthony Bourdain: Parts Unknown,NaN
6795,s6796,TV Show,Frasier,NaN
6806,s6807,TV Show,Friends,NaN
6901,s6902,TV Show,Gunslinger Girl,NaN



rating: 4 missing values


,show_id,type,title,rating
5989,s5990,Movie,13TH: A Conversation with Oprah Winfrey & Ava DuVernay,NaN
6827,s6828,TV Show,Gargantia on the Verdurous Planet,NaN
7312,s7313,TV Show,Little Lunch,NaN
7537,s7538,Movie,My Honor Was Loyalty,NaN



duration: 3 missing values


,show_id,type,title,duration
5541,s5542,Movie,Louis C.K. 2017,NaN
5794,s5795,Movie,Louis C.K.: Hilarious,NaN
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,NaN


In [28]:
df.loc[df["rating"].isna(), ["show_id", "type", "title", "release_year", "duration"]]

,show_id,type,title,release_year,duration
5989,s5990,Movie,13TH: A Conversation with Oprah Winfrey & Ava DuVernay,2017,37 min
6827,s6828,TV Show,Gargantia on the Verdurous Planet,2013,1 Season
7312,s7313,TV Show,Little Lunch,2015,1 Season
7537,s7538,Movie,My Honor Was Loyalty,2015,115 min


In [29]:
df.loc[df["duration"].isna(), ["show_id", "type", "title", "rating"]]

,show_id,type,title,rating
5541,s5542,Movie,Louis C.K. 2017,74 min
5794,s5795,Movie,Louis C.K.: Hilarious,84 min
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,66 min


For this project, we'll use the following approach:

director, cast, and country: retain the records and label missing values as "Unknown" so they're distinguishable from actual values.

In [30]:
df["date_added"] = pd.to_datetime(
    df["date_added"],
    format="mixed",
    errors="coerce"
)

df["date_added"].dtype

dtype('<M8[us]')

duration mixed data

#checking missing ratings

In [31]:
df.loc[
    df["rating"].isna(),
    ["show_id", "type", "title", "release_year", "duration"]
]

,show_id,type,title,release_year,duration
5989,s5990,Movie,13TH: A Conversation with Oprah Winfrey & Ava DuVernay,2017,37 min
6827,s6828,TV Show,Gargantia on the Verdurous Planet,2013,1 Season
7312,s7313,TV Show,Little Lunch,2015,1 Season
7537,s7538,Movie,My Honor Was Loyalty,2015,115 min


In [32]:
df.loc[
    df["duration"].isna(),
    ["show_id", "type", "title", "rating"]
]

,show_id,type,title,rating
5541,s5542,Movie,Louis C.K. 2017,74 min
5794,s5795,Movie,Louis C.K.: Hilarious,84 min
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,66 min


dataset has mixed data in the rating column.

In [33]:
# Identify ratings that incorrectly contain movie durations
misplaced_duration = df["rating"].str.fullmatch(
    r"\s*\d+\s*min\s*",
    na=False
)

# Move the duration into the correct column
df.loc[misplaced_duration, "duration"] = df.loc[
    misplaced_duration, "rating"
]

# Mark the invalid ratings as missing
df.loc[misplaced_duration, "rating"] = pd.NA

# Review the corrected records
df.loc[
    misplaced_duration,
    ["show_id", "title", "rating", "duration"]
]

,show_id,title,rating,duration
5541,s5542,Louis C.K. 2017,NaN,74 min
5794,s5795,Louis C.K.: Hilarious,NaN,84 min
5813,s5814,Louis C.K.: Live at the Comedy Store,NaN,66 min


rebuilding the dataset

In [34]:
duration_parts = df["duration"].str.extract(
    r"^\s*(\d+)\s*(min|season|seasons)\s*$"
)

df["duration_value"] = pd.to_numeric(
    duration_parts[0],
    errors="coerce"
)

df["duration_unit"] = duration_parts[1].str.lower()

df.loc[
    ["s5542", "s5795", "s5814"] if False else df["show_id"].isin(["s5542", "s5795", "s5814"]),
    ["show_id", "title", "rating", "duration", "duration_value", "duration_unit"]
]

,show_id,title,rating,duration,duration_value,duration_unit
5541,s5542,Louis C.K. 2017,NaN,74 min,74.0,min
5794,s5795,Louis C.K.: Hilarious,NaN,84 min,84.0,min
5813,s5814,Louis C.K.: Live at the Comedy Store,NaN,66 min,66.0,min


In [35]:
print("Dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())

quality_report = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
}).sort_values("missing_count", ascending=False)

display(quality_report)

Dataset shape: (8807, 14)
Duplicate rows: 0


,data_type,missing_count,missing_percent,unique_values
duration_unit,str,2676,30.38,1
duration_value,float64,2676,30.38,205
director,str,2634,29.91,4528
country,str,831,9.44,748
cast,str,825,9.37,7692
date_added,datetime64[us],10,0.11,1714
rating,str,7,0.08,14
title,str,0,0.00,8807
show_id,str,0,0.00,8807
type,str,0,0.00,2


In [36]:
text_columns = ["director", "cast", "country"]

for column in text_columns:
    df[column] = df[column].fillna("Unknown")

df[text_columns].isna().sum()

director    0
cast        0
country     0
dtype: int64

replace missing values with "Unknown" while preserving all rows.

distinguishing a movie's running time in minutes from a TV show's number of seasons.

In [37]:
duration_parts = df["duration"].str.extract(
    r"^\s*(\d+)\s*(min|season|seasons)\s*$",
    flags=0
)

df["duration_value"] = pd.to_numeric(
    duration_parts[0],
    errors="coerce"
)

df["duration_unit"] = duration_parts[1].str.lower()

df[
    ["title", "type", "duration", "duration_value", "duration_unit"]
].head(10)

,title,type,duration,duration_value,duration_unit
0,Dick Johnson Is Dead,Movie,90 min,90.0,min
1,Blood & Water,TV Show,2 Seasons,NaN,NaN
2,Ganglands,TV Show,1 Season,NaN,NaN
3,Jailbirds New Orleans,TV Show,1 Season,NaN,NaN
4,Kota Factory,TV Show,2 Seasons,NaN,NaN
5,Midnight Mass,TV Show,1 Season,NaN,NaN
6,My Little Pony: A New Generation,Movie,91 min,91.0,min
7,Sankofa,Movie,125 min,125.0,min
8,The Great British Baking Show,TV Show,9 Seasons,NaN,NaN
9,The Starling,Movie,104 min,104.0,min
